## Overview

This notebook validates the data pipeline for the Decision Intelligence Dashboard created by Member 3. It programmatically queries Google BigQuery to ensure the frontend KPIs, coordinate boundaries, and Top-10 inspection tables perfectly match the backend risk model.

---
## Input

BigQuery Dataset:
- `dengue_ew`

Input Table:
- `features` / `inspection_priority`

---
## Processing Steps

1. Connect to Google BigQuery.
2. Validate KPI telemetry (Max score, total cell count).
3. Validate geospatial bounds (Lat/Lon ranges).
4. Simulate the Dashboard Top-10 Sorting Logic.
5. Document UI/UX design rules.

In [ ]:
import os
from google.cloud import bigquery
import pandas as pd

# 1. Connect to Google BigQuery
project_id = 'dengue-early-warning'
dataset_id = 'dengue_ew'
client = bigquery.Client(project=project_id)

print("Successfully connected to Data Warehouse.")

In [ ]:
# 2 & 3. Validate KPI Telemetry & Geospatial Bounds
kpi_query = f"""
    SELECT 
        COUNT(DISTINCT cell_id) as total_cells,
        MAX(case_density_14d) as current_max_risk,
        MIN(lat) as min_lat, MAX(lat) as max_lat,
        MIN(lon) as min_lon, MAX(lon) as max_lon
    FROM `{project_id}.{dataset_id}.features`
"""
df_kpis = client.query(kpi_query).to_dataframe()

print("--- DASHBOARD KPI PARITY CHECK ---")
print(f"Dashboard Cell Count matches (1,759): {df_kpis['total_cells'].values[0] == 1759}")
print(f"Dashboard Max Score matches (37): {df_kpis['current_max_risk'].values[0] == 37}")
print(f"Map Bounds Confirmed: Lat({df_kpis['min_lat'].values[0]:.2f} to {df_kpis['max_lat'].values[0]:.2f})")

In [ ]:
# 4. Simulate the Dashboard Top-10 Sorting Logic
triage_query = f"""
    SELECT 
        cell_id, 
        lat, 
        lon, 
        case_density_14d
    FROM `{project_id}.{dataset_id}.features`
    ORDER BY case_density_14d DESC
    LIMIT 10
"""
df_top10 = client.query(triage_query).to_dataframe()

print("\n--- TOP 10 HIGH-RISK DASHBOARD PREVIEW ---")
display(df_top10)

---
## Result

Data pipeline validated. The backend BigQuery metrics perfectly align with the interactive Looker Studio visualizations.

---
## Author
Member 3
UI/UX & Dashboard Analytics
Gen AI Academy APAC